In [ ]:
import sys, os
from pathlib import Path

parent_folder = str(Path.cwd().parents[2])
if parent_folder not in sys.path:
    sys.path.append(parent_folder)
    
from sigpy import mri
import scipy
import pickle
from sklearn.decomposition import PCA
from matplotlib.colors import ListedColormap
import seaborn as sns
import sigpy as sp
import cupy as cp
import numpy as np
from sigpy.mri.app import L1WaveletRecon
from custom_recons.created_app import TotalVariationRecon_Custom, TotalVariationRecon_Stacked

## My files
import save_data_helpers
import recon_functions
import recon_plot_helpers
from gating_functions import golden_angle_coords_3d
import sigpy.plot as pl

### Load data

In [ ]:
data_bins = save_data_helpers.read_pickle('/home/lilianae/projects/naf_clean/load_data_clean/subject2_mid0082/data_bins_phil_gating.pkl')
dcf_bins = save_data_helpers.read_pickle('/home/lilianae/projects/naf_clean/recons/subject2_mid0082/dcf_ksp_phil_gating_512_5gates.pkl')
spoke_bins = save_data_helpers.read_pickle('/home/lilianae/projects/naf_clean/load_data_clean/subject2_mid0082/spoke_bins_phil_gating.pkl')
mps = save_data_helpers.read_pickle('/home/lilianae/projects/naf_clean/coils/subject2_mid0082/espirit_mps_full_res_ksp_512_ungated.pkl')


num_gates = len(data_bins)
data_bins_with_dcf = [None] * num_gates

## Create list of data_bins with dcf applied
for gate in range(num_gates):
    data_bins_with_dcf[gate] = data_bins[gate] * dcf_bins[gate]
    print(f'Data bins w/ dcf shape = {data_bins_with_dcf[gate].shape}')
    print(f'coords shape = {spoke_bins[gate].shape}')

### Select only certain coils for reconstruction

In [ ]:
ksp_gates_5coils = [None] * num_gates
for gate in range(num_gates):
    coils_select = [1, 8, 9, 10, 13]
    ksp_coil_select = [data_bins_with_dcf[gate][i] for i in coils_select]
    ksp_coil_select = np.stack(ksp_coil_select, axis=0)
    ksp_gates_5coils[gate] = ksp_coil_select
    print(f'ksp_coil_select.shape = {ksp_coil_select.shape}')


mps_coil_select = [mps[i] for i in coils_select]
mps_coil_select = np.stack(mps_coil_select, axis=0)
print(f'mps_coil_select.shape = {mps_coil_select.shape}')

In [ ]:
print(spoke_bins[0].shape)

### Try using assembled App

In [ ]:
device=2
## Get device

for i in range(5):
    print(f"\rPre-initialization: TV recon for gate {i}/{num_gates}", end='', flush=True)
    print()
    tv_preinit_alg =TotalVariationRecon_Stacked(y=ksp_gates_5coils[i],
                                                    mps=mps_coil_select,
                                                    lamda=1e-3, 
                                                    coord=spoke_bins[i],
                                                    device=device,
                                                    z=None, 
                                                    max_iter=50,
                                                    max_power_iter=50,
                                                    show_pbar=True)
    result = tv_preinit_alg.run()


In [ ]:
print(result.shape)

In [ ]:
result_app_cropped = recon_plot_helpers.crop_xy_dimension(result, oshape=(58, 256, 256))
fig, axs = recon_plot_helpers.plot_recons_all_axes(np.abs(cp.asnumpy(result_app_cropped)), title=f'Gate 0 - lam = 1e-2 - iters=100')